In [ ]:
!pip install tensorflow
!pip install tensorflow_datasets

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
# import seaborn as sns
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.image as mpimg
# import itertools

print(tf.__version__)

2.19.0


In [ ]:
cifar10 = tf.keras.datasets.cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
# Shuffle the training data first
indices = tf.random.shuffle(tf.range(len(x_train)))
x_train_shuffled = tf.gather(x_train, indices)
y_train_shuffled = tf.gather(y_train, indices)

# Split into training and validation (80/20)
val_split = 0.2
num_train = int(len(x_train_shuffled) * (1 - val_split))
x_train_final = x_train_shuffled[:num_train]
y_train_final = y_train_shuffled[:num_train]
x_val = x_train_shuffled[num_train:]
y_val = y_train_shuffled[num_train:]

print(f"Training set size: {len(x_train_final)}")
print(f"Validation set size: {len(x_val)}")
print(f"Test set size: {len(x_test)}")

Training set size: 40000
Validation set size: 10000
Test set size: 10000


In [ ]:
dataset, info = tfds.load("cifar10", as_supervised=True, with_info=True)
dataset_size = info.splits["train"].num_examples # 3670
class_names = info.features["label"].names # ["dandelion", "daisy", ...]
n_classes = info.features["label"].num_classes # 5

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cifar10/incomplete.N8F54D_3.0.2/cifar10-train.tfrecord*...:   0%|         …

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cifar10/incomplete.N8F54D_3.0.2/cifar10-test.tfrecord*...:   0%|          …

Dataset cifar10 downloaded and prepared to /root/tensorflow_datasets/cifar10/3.0.2. Subsequent calls will reuse this data.


In [ ]:
sample_images = (x_train[:4].astype('float32')) / 255.0  # Normalize to 0-1
# Resize to 299x299 for Xception
images_resized = tf.image.resize(sample_images, (299, 299))

In [ ]:
inputs = tf.keras.applications.xception.preprocess_input(images_resized)

In [ ]:
batch_size = 32
preprocess = tf.keras.Sequential([
tf.keras.layers.Resizing(height=299, width=299, crop_to_aspect_ratio=True),
tf.keras.layers.Lambda(tf.keras.applications.xception.preprocess_input)
])

train_set = tf.data.Dataset.from_tensor_slices((x_train_final, y_train_final)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)
valid_set = tf.data.Dataset.from_tensor_slices((x_val, y_val)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)
test_set = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)

In [ ]:
data_augmentation = tf.keras.Sequential([
tf.keras.layers.RandomFlip(mode="horizontal", seed=42),
tf.keras.layers.RandomRotation(factor=0.05, seed=42),
tf.keras.layers.RandomContrast(factor=0.2, seed=42)
])


In [ ]:
# ==============================================================================
# PHASE 1: REPRESENTATION EXTRACTION & ARCHITECTURE SETUP
# ==============================================================================
# Instantiate the base model (Xception) with weights pre-trained on ImageNet.
# We set include_top=False to discard the original 1000-class classifier head, 
# as we only want the convolutional base to act as a "feature extractor".
base_model = tf.keras.applications.xception.Xception(weights="imagenet", include_top=False)

# Define the input shape. Xception was designed for 299x299 RGB images.
inputs = tf.keras.Input(shape=(299, 299, 3))

# ==============================================================================
# IN-MODEL DATA AUGMENTATION
# ==============================================================================
# Pass inputs through our augmentation block. By putting this inside the model,
# Keras automatically applies it ONLY during training (not during validation/testing)
# and it benefits from GPU acceleration. This prevents overfitting.
x = data_augmentation(inputs)

# ==============================================================================
# THE FROZEN BASE
# ==============================================================================
# Pass the augmented images through the base model. 
# CRITICAL: training=False ensures that BatchNormalization layers don't update 
# their internal statistics based on our new dataset, keeping the pre-trained weights stable.
x = base_model(x, training=False)

# ==============================================================================
# THE CUSTOM HEAD
# ==============================================================================
# GlobalAveragePooling2D flattens the spatial dimensions (turns feature maps into a 1D vector).
x = tf.keras.layers.GlobalAveragePooling2D()(x)
# Dense layer is our new classifier. n_classes=10 for CIFAR-10. Softmax gives probability distribution.
outputs = tf.keras.layers.Dense(n_classes, activation="softmax")(x)

# Create the final compiled Model connecting inputs to outputs.
Model = tf.keras.Model(inputs=inputs, outputs=outputs)

# Freeze ALL layers in the base model. We do not want backpropagation to change
# these pre-trained "ImageNet" weights during the first phase of training.
for layer in base_model.layers:
    layer.trainable = False


83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
# ==============================================================================
# TRAINING PHASE 1: WARMING UP THE HEAD
# ==============================================================================
# Since our new Dense head has random weights, we use a relatively high 
# learning rate (0.1) to make large updates ("rough blocking").
# The rest of the model is frozen, so this only trains the new classification layer.
optimizer = tf.keras.optimizers.SGD(learning_rate=0.1, momentum=0.9)
Model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

# Train for a few epochs to get the head's weights into the right ballpark.
history = Model.fit(train_set, validation_data=valid_set, epochs=3)


Epoch 1/3
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 538s 422ms/step - accuracy: 0.6508 - loss: 1.1689 - val_accuracy: 0.7967 - val_loss: 0.7210
Epoch 2/3
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 535s 428ms/step - accuracy: 0.6867 - loss: 1.0639 - val_accuracy: 0.7849 - val_loss: 0.7277
Epoch 3/3
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 535s 428ms/step - accuracy: 0.6999 - loss: 1.0206 - val_accuracy: 0.8285 - val_loss: 0.5804


In [ ]:
# ==============================================================================
# PHASE 2: INITIAL FINE-TUNING (UNFREEZING THE TAIL)
# ==============================================================================
# Now that the head is stable, we unfreeze the "later" layers of the base model 
# (from layer 56 onwards). Earlier layers learn generic edges/colors, but these 
# later layers learn abstract, domain-specific concepts that we want to adapt to CIFAR-10.
for layer in base_model.layers[56:]:
    layer.trainable = True


In [ ]:
# ==============================================================================
# TRAINING PHASE 2: CAREFUL NUDGING
# ==============================================================================
# CRITICAL: We must re-compile the model for the unfreezing to take effect.
# We also drastically drop the learning rate to 1e-4. We only want to "nudge" 
# the pre-trained weights. A large learning rate here would cause "Catastrophic Forgetting."
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-4, momentum=0.9)
Model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

# Train the unfrozen layers and the head jointly.
history = Model.fit(train_set, validation_data=valid_set, epochs=10)


Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1064s 842ms/step - accuracy: 0.6951 - loss: 0.9138 - val_accuracy: 0.8264 - val_loss: 0.5231
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1050s 840ms/step - accuracy: 0.7650 - loss: 0.6831 - val_accuracy: 0.8520 - val_loss: 0.4410
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1049s 839ms/step - accuracy: 0.7935 - loss: 0.6029 - val_accuracy: 0.8669 - val_loss: 0.3950
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1046s 837ms/step - accuracy: 0.8085 - loss: 0.5558 - val_accuracy: 0.8756 - val_loss: 0.3715
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1049s 839ms/step - accuracy: 0.8232 - loss: 0.5127 - val_accuracy: 0.8813 - val_loss: 0.3512
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1051s 841ms/step - accuracy: 0.8354 - loss: 0.4792 - val_accuracy: 0.8869 - val_loss: 0.3380
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1050s 840ms/step - accuracy: 0.8461 - loss: 0.4474 - val_accuracy: 0.8871 - val_loss: 0.3325
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1049s 840ms/s

In [19]:
# ==============================================================================
# PHASE 3: EXPERIMENT 1 - DEEPER FINE-TUNING & ADAPTIVE LR
# ==============================================================================
# We grant the model more "plasticity" by unfreezing deeper into the network 
# (starting from layer 30). This allows middle-level feature detectors (like textures) 
# to adapt to the low-resolution nature of CIFAR-10.
for layer in base_model.layers[30:]:
    layer.trainable = True

# Setup the Learning Rate Decay Callback (The "Thermostat")
# If validation loss (our true error metric) stops improving for 2 epochs, 
# multiply the learning rate by 0.2. This forces the optimizer to take smaller, 
# more precise steps to find the global minimum.
lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2, 
    patience=2,  
    min_lr=1e-6, 
    verbose=1
)

# Re-compile to apply the unfreezing of layer 30+.
# Start with our safe, small learning rate of 1e-4.
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-4, momentum=0.9)
Model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

# Train for 15 epochs, allowing the callback to automatically step down the LR when needed.
print("Starting the deeper fine-tuning experiment with learning rate decay...")
exp_history = Model.fit(
    train_set,
    validation_data=valid_set,
    epochs=15,
    callbacks=[lr_reducer]
)


Starting the deeper fine-tuning experiment with learning rate decay...
Epoch 1/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1291s 1s/step - accuracy: 0.7777 - loss: 0.6664 - val_accuracy: 0.8714 - val_loss: 0.3962 - learning_rate: 1.0000e-04
Epoch 2/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1283s 1s/step - accuracy: 0.8284 - loss: 0.4946 - val_accuracy: 0.8813 - val_loss: 0.3579 - learning_rate: 1.0000e-04
Epoch 3/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1282s 1s/step - accuracy: 0.8487 - loss: 0.4336 - val_accuracy: 0.8904 - val_loss: 0.3326 - learning_rate: 1.0000e-04
Epoch 4/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1277s 1s/step - accuracy: 0.8638 - loss: 0.3947 - val_accuracy: 0.8960 - val_loss: 0.3205 - learning_rate: 1.0000e-04
Epoch 5/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1277s 1s/step - accuracy: 0.8774 - loss: 0.3600 - val_accuracy: 0.9001 - val_loss: 0.3123 - learning_rate: 1.0000e-04
Epoch 6/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1275s 1s/step - accuracy: 0.8826 - loss: 0.3359 - val_accuracy: 0.9028 - val_loss: 0.30

In [20]:
# ==============================================================================
# PHASE 4: FINAL AGGRESSIVE OPTIMIZATION
# ==============================================================================
# We use a slightly more patient callback so it doesn't reduce the LR too quickly.
lr_reducer_aggressive = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor= 0.2,
    patience=3, 
    min_lr=1e-7,
    verbose=1
)

# Re-compile with an "aggressive" learning rate of 1e-3.
# This gives the model enough energy to escape local minima plateaus, 
# but risks causing overfitting (memorization) if left unchecked.
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-3, momentum=0.9)
Model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
print("Starting the aggressive fine-tuning (LR=1e-3) experiment with learning rate decay...")

# Train for 21 epochs. (Note: Our logs showed this caused significant overfitting, 
# where training accuracy hit 99% but validation accuracy plateaued).
final_history = Model.fit(
    train_set,
    validation_data=valid_set,
    epochs=21,
    callbacks=[lr_reducer_aggressive]
)

# Final evaluation on the unseen test set to measure true generalization.
test_loss, test_acc = Model.evaluate(test_set)

# FIXED SYNTAX BUG HERE (* instead of :)
print(f"Final Optimized Test Accuracy: {test_acc * 100:.2f}%")


Starting the aggressive fine-tuning (LR=1e-3) experiment with learning rate decay...
Epoch 1/21
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1290s 1s/step - accuracy: 0.9229 - loss: 0.2188 - val_accuracy: 0.9030 - val_loss: 0.3290 - learning_rate: 0.0010
Epoch 2/21
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1314s 1s/step - accuracy: 0.9544 - loss: 0.1368 - val_accuracy: 0.9185 - val_loss: 0.2875 - learning_rate: 0.0010
Epoch 3/21
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1274s 1s/step - accuracy: 0.9739 - loss: 0.0840 - val_accuracy: 0.9207 - val_loss: 0.2908 - learning_rate: 0.0010
Epoch 4/21
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1272s 1s/step - accuracy: 0.9833 - loss: 0.0567 - val_accuracy: 0.9163 - val_loss: 0.3116 - learning_rate: 0.0010
Epoch 5/21
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1271s 1s/step - accuracy: 0.9887 - loss: 0.0404 - val_accuracy: 0.9285 - val_loss: 0.2767 - learning_rate: 0.0010
Epoch 6/21
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1270s 1s/step - accuracy: 0.9918 - loss: 0.0310 - val_accuracy: 0.9278 - val_loss: 0.2733 - l

ValueError: Invalid format specifier '100:.2f' for object of type 'float'